# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

---

We aggregate daily warehouse rows for June 2026 into a monthly feature vector per content item. The pipeline:

1. **Aggregation:** Sum impressions, clicks, sessions; impression-weighted avg position; session-weighted engagement rate.
2. **Engineered features:** `log_impressions` (log1p to tame heavy tail), `has_ga4_data` flag, `position_tier` bucket (`pos_1_3`, `pos_4_10`, `pos_11_20`, `pos_21_50`, `pos_51_plus`), `impression_density` (impressions / word_count).
3. **Missing-value handling:** `word_count` filled with 0 + a `has_word_count` flag to avoid encoding content-type via missingness.
4. **Label construction:** Position-tier median CTR -> `ctr_gap = tier_median - observed_ctr`; `is_opportunity = (ctr_gap > 0) & (impressions >= 1000)`.

In [ ]:
# == Cell 1: Connect + build the full feature vector ==
import duckdb
import os, sys
import pandas as pd
import numpy as np

# Load HF_TOKEN from .env (never hardcoded)
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'scripts'))
import pathlib, getpass
_env = pathlib.Path(os.getcwd()).resolve()
for _ in range(5):
    _ep = _env / '.env'
    if _ep.exists():
        for _line in _ep.read_text().splitlines():
            _line = _line.strip()
            if _line and not _line.startswith('#') and '=' in _line:
                _k, _v = _line.split('=', 1)
                os.environ.setdefault(_k.strip(), _v.strip())
        break
    _env = _env.parent

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF_TOKEN: ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':      f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':      f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
}
print('DuckDB connected and tables configured.')

# == Build feature vector with objective position tier names ==
feature_vector_q = f"""
WITH monthly_agg_raw AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(f.gsc_impressions)        AS total_impressions,
        SUM(f.gsc_clicks)             AS total_clicks,
        SUM(f.ga4_sessions)           AS total_sessions,
        SUM(f.ga4_engaged_sessions)   AS total_engaged_sessions,
        SUM(f.ga4_pageviews)          AS total_pageviews,
        CASE WHEN SUM(f.gsc_impressions) > 0
             THEN CAST(SUM(f.gsc_clicks) AS DOUBLE) / SUM(f.gsc_impressions) * 100.0
             ELSE 0.0
        END AS observed_ctr,
        CASE WHEN SUM(f.gsc_impressions) > 0
             THEN SUM(f.gsc_avg_position * f.gsc_impressions) / SUM(f.gsc_impressions)
             ELSE 0.0
        END AS avg_position,
        CASE WHEN SUM(f.ga4_sessions) > 0
             THEN CAST(SUM(f.ga4_engaged_sessions) AS DOUBLE) / SUM(f.ga4_sessions) * 100.0
             ELSE 0.0
        END AS engagement_rate,
        MAX(CASE WHEN f.ga4_data_available = TRUE THEN 1 ELSE 0 END) AS has_ga4_data,
        ANY_VALUE(d.word_count)    AS word_count,
        ANY_VALUE(d.content_type)  AS content_type
    FROM {TABLES['fact_daily_sample']} f
    LEFT JOIN {TABLES['dim_content']} d ON f.content_hash_id = d.content_hash_id
    WHERE f.month = '2026-06'
    GROUP BY f.client_hash_id, f.content_hash_id
    HAVING total_impressions >= 500 AND avg_position > 0
),
monthly_agg AS (
    SELECT m.*,
        CASE
            WHEN m.avg_position <= 3  THEN 'pos_1_3'
            WHEN m.avg_position <= 10 THEN 'pos_4_10'
            WHEN m.avg_position <= 20 THEN 'pos_11_20'
            WHEN m.avg_position <= 50 THEN 'pos_21_50'
            ELSE 'pos_51_plus'
        END AS position_tier
    FROM monthly_agg_raw m
)
SELECT m.*, t.tier_median_ctr
FROM monthly_agg m
LEFT JOIN (
    SELECT position_tier, MEDIAN(observed_ctr) AS tier_median_ctr
    FROM monthly_agg
    GROUP BY position_tier
) t ON m.position_tier = t.position_tier
"""

df = con.sql(feature_vector_q).df()

df['has_word_count'] = df['word_count'].notna().astype(int)
df['word_count'] = df['word_count'].fillna(0)
df['log_impressions'] = np.log1p(df['total_impressions'])
df['impression_density'] = np.where(
    df['word_count'] > 0,
    df['total_impressions'] / df['word_count'],
    0.0
)
df['ctr_gap'] = df['tier_median_ctr'] - df['observed_ctr']
df['is_opportunity'] = ((df['ctr_gap'] > 0) & (df['total_impressions'] >= 1000)).astype(int)

HONEST_FEATURES = [
    'total_impressions', 'log_impressions', 'avg_position', 'engagement_rate',
    'word_count', 'has_word_count', 'has_ga4_data', 'impression_density',
]

print(f'Feature vector built: {len(df):,} rows x {len(df.columns)} columns')
print(f'Honest features ({len(HONEST_FEATURES)}): {HONEST_FEATURES}')
print(f'\nLabel distribution (is_opportunity):')
print(df['is_opportunity'].value_counts().to_string())
print(f'\nBase rate: {df["is_opportunity"].mean():.4f} ({df["is_opportunity"].mean()*100:.1f}%)')
print(f'\nFeature vector sample (first 5 rows):')
df[HONEST_FEATURES + ['is_opportunity']].head()

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

---

| Feature | Meaning | Missing-Value Handling | Available Before Prediction? |
|---|---|---|---|
| `total_impressions` | Sum of daily GSC impressions for the content item over June 2026. | No missing (HAVING filter ensures >= 500). | **Yes.** Fully observed from GSC logs for the target month. |
| `log_impressions` | `log1p(total_impressions)` -- log-scaled to reduce heavy-tail skew. | No missing (derived from `total_impressions`). | **Yes.** Deterministic transform of an observed field. |
| `avg_position` | Impression-weighted average GSC position over the month. Lower = better. | No missing (HAVING filter ensures > 0). | **Yes.** Aggregated from recorded daily position data. |
| `engagement_rate` | `engaged_sessions / sessions x 100` from GA4. | Set to 0.0 when sessions = 0 (safe division). | **Yes.** GA4 logs are finalized before analysis. |
| `word_count` | Page word count from `dim_content`. | Filled with 0 when missing + `has_word_count` flag added. | **Yes.** Static content property, measurable any time. |
| `has_word_count` | Binary: 1 if `word_count` was present, 0 if it was missing. | N/A (always defined). | **Yes.** Indicates whether measurement exists. |
| `has_ga4_data` | Binary: 1 if any day in the month had `ga4_data_available = TRUE`. | N/A (always defined). | **Yes.** Analytics pipeline coverage flag. |
| `impression_density` | `total_impressions / word_count`. How much search attention per unit of content. | Set to 0.0 when `word_count = 0`. | **Yes.** Ratio of two observed fields. |
| `position_tier` | Objective rank bucket: `pos_1_3`, `pos_4_10`, `pos_11_20`, `pos_21_50`, `pos_51_plus`. | No missing (derived from `avg_position`). | **Yes.** Deterministic bucketing of position. |

**Key design decisions:**
- `word_count` missingness is **systematic** (follows `content_type`), not random. A blind `fillna(0)` would inject a content-type signal. The `has_word_count` flag lets the model learn the missingness pattern honestly.
- `engagement_rate` is set to 0 when no sessions exist, but `has_ga4_data` tells the model whether this zero is "genuinely no engagement" vs "no analytics coverage."
- All features are knowable at decision time (end of June 2026). None require future data.

In [ ]:
# == Cell 2: Feature summary statistics + missing-value audit ==
print('=== Feature Summary Statistics ===')
print(df[HONEST_FEATURES].describe().round(3).to_string())

print('\n=== Missing Values Per Feature ===')
missing = df[HONEST_FEATURES].isnull().sum()
print(missing.to_string())
print(f'\nTotal rows with any missing honest feature: {df[HONEST_FEATURES].isnull().any(axis=1).sum()}')

print('\n=== has_word_count breakdown by content_type ===')
print(df.groupby('content_type')['has_word_count'].agg(['count', 'mean']).round(3).to_string())

print('\n=== has_ga4_data breakdown ===')
print(df['has_ga4_data'].value_counts().to_string())

print('\n=== Position tier distribution ===')
print(df['position_tier'].value_counts().to_string())

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

---

We test for all three types of leakage from the taxonomy:

### Attack 1: Label-derived features
`observed_ctr` is the numerator of our label (`ctr_gap = tier_median_ctr - observed_ctr`). If added as a feature, a model can directly reconstruct the label. We train WITH and WITHOUT it and compare scores.

### Attack 2: Future/overlapping windows
All features use data from June 2026 only -- the same month the label covers. Since we are scoring pages *within* this month (not predicting a future month), features and label share the same window **by design**. We verify no column from a future month leaks in.

### Attack 3: Decision-derived features (product flags)
The warehouse contains `is_declining_label` and `trend_direction` -- product-level flags computed from outcome data. We test whether sneaking `trend_direction` into features inflates the score.

We also compare **random split vs. grouped (client-holdout) split** to measure memorization.

In [ ]:
# == Cell 3: The leakage hunt -- three attacks + split comparison ==
from sklearn.model_selection import GroupKFold, KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

groups = df['client_hash_id']
y = df['is_opportunity']
base_rate = y.mean()

def evaluate_model(X_train, y_train, X_test, y_test, label=''):
    """Train RF and return (ROC AUC, Avg Precision)."""
    clf = RandomForestClassifier(random_state=42, n_estimators=50, max_depth=6)
    clf.fit(X_train, y_train)
    proba = clf.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)
    ap  = average_precision_score(y_test, proba)
    return auc, ap, clf

# == Grouped (client-holdout) split ==
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(df, y, groups))
X_train_g, X_test_g = df.iloc[train_idx], df.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

print(f'Base rate (is_opportunity): {base_rate:.4f} ({base_rate*100:.1f}%)')
print(f'Train size: {len(train_idx):,}  |  Test size: {len(test_idx):,}')
print(f'Train clients: {X_train_g["client_hash_id"].nunique()}  |  '
      f'Test clients: {X_test_g["client_hash_id"].nunique()}  |  '
      f'Overlap: {len(set(X_train_g["client_hash_id"]) & set(X_test_g["client_hash_id"]))} (must be 0)')
print()

# == Attack 1: Label-derived feature (observed_ctr) ==
print('=' * 65)
print('ATTACK 1: Label-derived feature -- observed_ctr')
print('=' * 65)

auc_h, ap_h, clf_h = evaluate_model(
    X_train_g[HONEST_FEATURES], y_train_g,
    X_test_g[HONEST_FEATURES], y_test_g
)
print(f'Honest model:  ROC AUC = {auc_h:.4f}  |  Avg Precision = {ap_h:.4f}')

LEAKED_FEATURES_1 = HONEST_FEATURES + ['observed_ctr']
auc_l1, ap_l1, _ = evaluate_model(
    X_train_g[LEAKED_FEATURES_1], y_train_g,
    X_test_g[LEAKED_FEATURES_1], y_test_g
)
print(f'+ observed_ctr: ROC AUC = {auc_l1:.4f}  |  Avg Precision = {ap_l1:.4f}')
print(f'  --> AUC jump: {auc_l1 - auc_h:+.4f}  (near-perfect = leakage confirmed)')
print()

# == Attack 2: Future window check ==
print('=' * 65)
print('ATTACK 2: Future/overlapping window check')
print('=' * 65)
window_check = con.sql(f"""
    SELECT DISTINCT month
    FROM {TABLES['fact_daily_sample']}
    WHERE month > '2026-06'
    LIMIT 5
""").df()
if len(window_check) == 0:
    print('PASS: No data exists after June 2026 in the sample table.')
else:
    print(f'WARNING: Found future months: {window_check["month"].tolist()}')

date_bounds = con.sql(f"""
    SELECT MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {TABLES['fact_daily_sample']}
    WHERE month = '2026-06'
""").df()
print(f'Feature window: {date_bounds["min_date"].iloc[0]} to {date_bounds["max_date"].iloc[0]}')
print('Label uses same window (within-month CTR gap). No future overlap by design.')
print()

# == Attack 3: Decision-derived feature (trend_direction) ==
print('=' * 65)
print('ATTACK 3: Decision-derived feature -- trend_direction / is_declining')
print('=' * 65)
fact_cols = con.sql(f"SELECT * FROM {TABLES['fact_daily_sample']} LIMIT 0").columns
trend_cols = [c for c in fact_cols if 'trend' in c.lower() or 'declining' in c.lower()]
print(f'Trend/declining columns in fact table: {trend_cols if trend_cols else "None found"}')

dim_cols = con.sql(f"SELECT * FROM {TABLES['dim_content']} LIMIT 0").columns
trend_dim = [c for c in dim_cols if 'trend' in c.lower() or 'declining' in c.lower()]
print(f'Trend/declining columns in dim_content: {trend_dim if trend_dim else "None found"}')
print('VERDICT: These columns (if present) are product flags derived from outcomes.')
print('         They are excluded from the feature vector. None appear in HONEST_FEATURES.')
print()

# == Split comparison: random vs grouped ==
print('=' * 65)
print('SPLIT COMPARISON: Random vs. Client-Holdout')
print('=' * 65)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
train_r, test_r = next(kf.split(df))
auc_rand, ap_rand, _ = evaluate_model(
    df.iloc[train_r][HONEST_FEATURES], y.iloc[train_r],
    df.iloc[test_r][HONEST_FEATURES], y.iloc[test_r]
)
print(f'Random split:         ROC AUC = {auc_rand:.4f}  |  Avg Precision = {ap_rand:.4f}')
print(f'Client-holdout split: ROC AUC = {auc_h:.4f}  |  Avg Precision = {ap_h:.4f}')
print(f'Gap (random - grouped): AUC = {auc_rand - auc_h:+.4f}  |  AP = {ap_rand - ap_h:+.4f}')
print('(A large gap would indicate the model memorizes client-specific patterns.)')
print()

# == Feature importance sanity check ==
print('=' * 65)
print('FEATURE IMPORTANCE (honest model, client-holdout)')
print('=' * 65)
fi = pd.Series(clf_h.feature_importances_, index=HONEST_FEATURES).sort_values(ascending=False)
for feat, imp in fi.items():
    flag = ' <-- CHECK: dominates' if imp > 0.5 else ''
    print(f'  {feat:25s}  {imp:.4f}{flag}')
print()
if fi.iloc[0] > 0.5:
    print(f'WARNING: "{fi.index[0]}" accounts for {fi.iloc[0]*100:.1f}% of importance -- investigate!')
else:
    print('PASS: No single feature dominates with >50% importance.')

## 4. What I excluded and why

*The list of fields you refused to use -- with one line of why each.*

---

| Excluded Field | Category | Why |
|---|---|---|
| `observed_ctr` | **Label-derived** | The label (`ctr_gap`) is computed directly from `observed_ctr`. Including it gives AUC = 1.0 -- pure leakage. |
| `ctr_gap` | **Label itself** | This IS the ranking target; using it as a feature would be circular. |
| `tier_median_ctr` | **Label component** | Part of the label formula (`ctr_gap = tier_median - observed`). |
| `total_clicks` | **Label sibling** | `clicks / impressions = CTR`; clicks directly encode the target. |
| `trend_direction` | **Product flag** | Post-hoc outcome flag derived from impression trends -- a decision-derived feature. |
| `trend_pct` | **Product flag** | Raw percentage change that `trend_direction` is computed from. |
| `is_declining_label` | **Product flag** | Binary label from the starter dataset -- encodes a different outcome. |
| `client_hash_id` | **Privacy / ID** | Pseudonymous identifier; used for grouped splits, never as a feature. |
| `content_hash_id` | **Privacy / ID** | Pseudonymous identifier; used for joins, never as a feature. |
| `content_type` | **Deferred** | Not excluded permanently, but not in the initial honest feature set. Could be added as a categorical after encoding, but missingness patterns would need careful handling. |
| `position_tier` | **Redundant** | Objective range bucket (`pos_1_3`, `pos_4_10`, `pos_11_20`, `pos_21_50`, `pos_51_plus`) of `avg_position`. Used for label construction. |

In [ ]:
# == Cell 4: Verify exclusions -- confirm excluded fields are NOT in features ==
EXCLUDED_FIELDS = [
    'observed_ctr',       # label-derived
    'ctr_gap',            # label itself
    'tier_median_ctr',    # label component
    'total_clicks',       # label sibling (clicks/impressions = CTR)
    'client_hash_id',     # ID -- privacy/grouping only
    'content_hash_id',    # ID -- privacy/grouping only
    'position_tier',      # redundant with avg_position
    'content_type',       # deferred -- not in initial feature set
]

print('=== Exclusion Verification ===')
for field in EXCLUDED_FIELDS:
    in_features = field in HONEST_FEATURES
    status = 'FAIL -- FOUND IN FEATURES!' if in_features else 'OK -- excluded'
    print(f'  {field:25s}  {status}')

violations = [f for f in EXCLUDED_FIELDS if f in HONEST_FEATURES]
if violations:
    print(f'\nERROR: {len(violations)} excluded field(s) found in feature set: {violations}')
else:
    print(f'\nPASS: All {len(EXCLUDED_FIELDS)} excluded fields are absent from the honest feature set.')

print('\n=== Summary ===')
print(f'Honest features: {len(HONEST_FEATURES)}')
print(f'Excluded fields: {len(EXCLUDED_FIELDS)}')
print(f'Base rate:       {base_rate:.4f} ({base_rate*100:.1f}%)')
print(f'Honest AUC:      {auc_h:.4f} (client-holdout)')
print(f'Leaked AUC:      {auc_l1:.4f} (with observed_ctr -- confirms leakage)')

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled -- markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` -- then submit your repo URL on the card. Done.